<a href="https://colab.research.google.com/github/stellamaclelland/ISOM-260-Work/blob/main/public/session-07/ISOM260_S7_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Session 7 Homework: Build Your Own Knowledge Agent

**ISOM 260: AI for Business** | Suffolk University | Prof. Hasan Arslan

---

In class, your agent read **CloudSync Solutions'** documents. It could answer questions about pricing, refund policies, and support tiers — because we gave it those documents.

Now it reads **yours.**

You'll pick a topic you actually care about, load real information, and build a knowledge agent that becomes an expert in *your* domain. Same RAG engine from class — your content, your questions, your expertise.

### What You'll Do

| Task | What | Time |
|------|------|------|
| **Task 1: Build** | Pick a domain, load 5+ documents, test your agent | ~45 min |
| **Task 2: Break** | Find 3 failure modes (outdated data, contradictions, gaps) | ~30 min |
| **Task 3: Evaluate** | Score your agent on 10 test questions | ~30 min |
| **Task 4: Design** | Write a real-world knowledge agent proposal | ~20 min |

**Total: ~2 hours**

---

**Submission:** Run all cells so output is visible → Share your Colab link (Anyone with link can view) → Paste link in Canvas.

### A Note on Using AI Tools

You’re welcome to use ChatGPT, Claude, Copilot, or any AI tool to help you with this homework — for brainstorming domain ideas, drafting document content, writing test questions, or polishing your proposal.

That’s not a loophole. That’s the skill. Knowing how to use AI tools effectively is literally what this course teaches. The learning happens when you choose a domain, decide what documents matter, evaluate whether the agent’s answers are actually correct, and figure out where it breaks.

No AI tool can do that evaluation for you — that requires *your* judgment.

---

## 🚀 Setup (Just Run These)

Same setup as class. 2 cells, 2 minutes. You’ve done this before.

In [1]:
# Install the Anthropic SDK
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 10.9 MB/s eta 0:00:00


In [2]:
# Set your API key + verify connection
import os
import json
import anthropic
from google.colab import userdata

# Load API key from Colab Secrets (recommended)
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace if needed
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

client = anthropic.Anthropic()

# Quick connection test
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Homework notebook connected!' and nothing else."}]
)
print(f"🟢 {response.content[0].text}")
print(f"   Model: {response.model}")

✅ API key loaded from Colab Secrets!
🟢 Homework notebook connected!
   Model: claude-sonnet-4-5-20250929


---

## 🔧 Your RAG Toolkit (Pre-Built)

These are the same functions from class. **Just run all 4 cells** — don’t modify them.

| Cell | What It Does |
|------|--------------|
| `chunk_documents()` | Splits your documents into searchable passages |
| `search_documents()` | Finds the most relevant passages for a query (TF-IDF) |
| `search_my_docs()` | Wrapper + tool definition so Claude can call it |
| `run_rag_agent()` | The full ReAct agent loop from class |

In [3]:
# ============================================
# DOCUMENT CHUNKING (from class)
# ============================================
# Splits documents into overlapping ~500-character chunks.
# Each chunk keeps its metadata so we know where it came from.

def chunk_documents(documents, chunk_size=500, overlap=100):
    """
    Split documents into overlapping chunks.

    Args:
        documents: List of dicts with 'title', 'content', 'source' fields
        chunk_size: Target size of each chunk in characters
        overlap: Number of overlapping characters between chunks

    Returns:
        List of chunk dicts with 'text', 'title', 'source', 'chunk_index' fields
    """
    chunks = []

    for doc in documents:
        content = doc['content']
        title = doc['title']
        source = doc['source']
        last_updated = doc.get('last_updated', 'Unknown')

        # Split into chunks with overlap
        start = 0
        chunk_index = 0

        while start < len(content):
            end = start + chunk_size

            # Try to break at a paragraph or sentence boundary
            if end < len(content):
                newline_pos = content.rfind('\n', start + chunk_size - 100, end + 50)
                if newline_pos > start:
                    end = newline_pos

            chunk_text = content[start:end].strip()

            if chunk_text:
                chunks.append({
                    'text': chunk_text,
                    'title': title,
                    'source': source,
                    'last_updated': last_updated,
                    'chunk_index': chunk_index
                })
                chunk_index += 1

            start = end - overlap if end < len(content) else len(content)

    return chunks

print("✅ chunk_documents() ready")

✅ chunk_documents() ready


In [4]:
# ============================================
# TF-IDF DOCUMENT SEARCH (from class)
# ============================================
# Scores each chunk by how relevant it is to a query.
# TF = how often a word appears in a chunk
# IDF = how rare that word is across ALL chunks

from collections import Counter
import math

def search_documents(query, chunks, top_k=3):
    """
    Search documents using TF-IDF scoring.
    Returns top_k most relevant (score, chunk) pairs.
    """
    stop_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'what', 'how',
        'do', 'does', 'i', 'my', 'our', 'their', 'for', 'to', 'and',
        'or', 'of', 'in', 'on', 'at', 'can', 'about', 'it', 'be',
        'that', 'this', 'with', 'from', 'not', 'but', 'if', 'when',
        'there', 'which', 'each', 'have', 'has', 'had', 'will', 'would'
    }

    query_terms = [w for w in query.lower().split() if w not in stop_words]
    if not query_terms:
        return []

    num_chunks = len(chunks)
    doc_freq = Counter()
    chunk_word_counts = []

    for chunk in chunks:
        words = chunk['text'].lower().split()
        word_count = Counter(words)
        chunk_word_counts.append(word_count)
        for term in set(words):
            doc_freq[term] += 1

    scored = []
    for idx, chunk in enumerate(chunks):
        word_count = chunk_word_counts[idx]
        total_words = sum(word_count.values()) or 1

        score = 0.0
        for term in query_terms:
            tf  = word_count[term] / total_words
            idf = math.log((num_chunks + 1) / (doc_freq.get(term, 0) + 1))
            score += tf * idf

        if score > 0:
            scored.append((round(score, 4), chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

print("✅ search_documents() ready — TF-IDF scoring")

✅ search_documents() ready — TF-IDF scoring


In [5]:
# ============================================
# SEARCH WRAPPER + TOOL DEFINITION
# ============================================
# This creates a search function and tool definition that reference
# the global `my_chunks` variable. You'll set that in Task 1.

def search_my_docs(query: str) -> str:
    """Search YOUR documents and return formatted results."""
    results = search_documents(query, my_chunks, top_k=3)

    if not results:
        return json.dumps({"message": "No relevant documents found. Try different search terms."})

    formatted = []
    for score, chunk in results:
        formatted.append({
            "relevance_score": round(score, 2),
            "document": chunk['title'],
            "source": chunk['source'],
            "last_updated": chunk['last_updated'],
            "content": chunk['text']
        })

    return json.dumps(formatted, indent=2)


# Tool definition — tells Claude how to call the search function
my_search_tool = {
    "name": "search_my_docs",
    "description": (
        "Search the knowledge base documents. Returns the most relevant "
        "passages for a query. Always search before answering questions."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use specific terms related to the question"
            }
        },
        "required": ["query"]
    }
}

my_tools = [my_search_tool]
my_tool_functions = {"search_my_docs": search_my_docs}

print("✅ search_my_docs() + tool definition ready")
print("   (Will search whatever documents you load in Task 1)")

✅ search_my_docs() + tool definition ready
   (Will search whatever documents you load in Task 1)


In [6]:
# ============================================
# THE RAG AGENT LOOP (from class)
# ============================================
# Same ReAct pattern: THINK → ACT → OBSERVE → REPEAT or CONCLUDE
# Uses MY_DOMAIN variable so the system prompt matches your topic.

def run_rag_agent(user_message: str, tools: list, tool_functions: dict, max_steps: int = 10):
    """
    Run a RAG knowledge agent using the ReAct framework.
    Searches documents before answering. Cites sources.
    """
    system_prompt = (
        f"You are a {MY_DOMAIN} knowledge agent using the ReAct framework.\n"
        f"You answer questions about {MY_DOMAIN} using ONLY the provided documents.\n\n"
        "For every question:\n"
        "1. THINK: What specific information do I need? Which document likely has it?\n"
        "2. ACT: Search documents with specific terms.\n"
        "3. OBSERVE: Did I find the relevant info? Is it current?\n"
        "4. REPEAT or CONCLUDE with a sourced answer.\n\n"
        "CRITICAL RULES:\n"
        "- ALWAYS search documents before answering questions\n"
        "- ONLY state facts found in documents — never make up information\n"
        "- If documents don't contain the answer, say: 'I don't have information on that in our current documents.'\n"
        "- Always cite which document your answer comes from\n"
        "- If you find conflicting information, flag it explicitly and note document dates\n"
        "- Pay attention to document dates — prefer the most recent version"
    )

    print(f"\n{'=' * 60}")
    print(f"💬 User: {user_message}")
    print(f"{'=' * 60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        print(f"\n{'- ' * 30}")
        print(f"🔄 Step {step} | Stop reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text" and block.text.strip():
                    print(f"\n   💭 THOUGHT:")
                    for line in block.text.strip().split('\n'):
                        print(f"      {line}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    print(f"\n   🎯 ACTION: {tool_name}")
                    print(f"      Input: {json.dumps(tool_input, indent=2)[:200]}")

                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                    print(f"\n   👁️  OBSERVATION:")
                    result_preview = result[:400] + "..." if len(result) > 400 else result
                    for line in result_preview.split('\n')[:10]:
                        print(f"      {line}")
                    if len(result) > 400:
                        print(f"      [...truncated for display, full data sent to agent]")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            print(f"\n{'=' * 60}")
            print(f"🤖 FINAL ANSWER:")
            print(f"{'=' * 60}")
            print(final_answer)
            print(f"\n✅ Done in {step} step(s)")

            return {"answer": final_answer, "messages": messages, "steps": step}

    print(f"\n⚠️ Max steps ({max_steps}) reached.")
    return {"answer": "Max steps reached.", "messages": messages, "steps": step}

print("✅ run_rag_agent() ready")
print("   Uses MY_DOMAIN variable for the system prompt.")

✅ run_rag_agent() ready
   Uses MY_DOMAIN variable for the system prompt.


---

## 📋 Task 1: Build Your Knowledge Agent (45 min)

Pick something you care about. A restaurant, a sports team, a club, a startup idea, your university, a hobby — anything with enough information to fill 5 documents.

| Domain Idea | Example Documents |
|---|---|
| 🍕 Restaurant | Menu, allergen info, hours, catering, reviews |
| ⚽ Sports Team | Roster, schedule, ticket prices, stadium info, history |
| 🚀 Startup | Product features, pricing, FAQ, team bios, roadmap |
| 🎓 University | Admissions, programs, campus life, financial aid, clubs |
| 🏋️ Gym / Studio | Classes, membership tiers, trainers, schedule, policies |
| 🎵 Music Artist | Discography, tour dates, bio, merch, fan FAQ |

### Step 1: Name Your Domain

In [7]:
# ============================================
# STEP 1: Name your domain
# ============================================
# Replace this with YOUR topic.

MY_DOMAIN = "Boston Bruins"  # <-- Change this!

print(f"🎯 Your domain: {MY_DOMAIN}")
print(f"   Your agent will become a {MY_DOMAIN} expert.")

🎯 Your domain: Boston Bruins
   Your agent will become a Boston Bruins expert.


### Step 2: Load Your Documents

Fill in **at least 5 documents** about your domain. Each document needs:
- `title` — what the document is about
- `source` — where it came from (can be made up, like "menu.pdf")
- `last_updated` — when it was last updated
- `content` — the actual text (aim for 100–500 words per document)

**Tips:**
- Quality > quantity. One detailed document beats five vague ones.
- Include specific facts: names, numbers, prices, dates, policies.
- Copy-paste from real sources (websites, menus, brochures) — that’s fine!
- Use AI to help draft content if you want. The learning is in testing, not typing.

**Document 1 is filled in as an example** (a restaurant menu). Replace it with your own content, then fill in the rest.

In [8]:
# ============================================
# STEP 2: Your documents
# ============================================
# Document 1 is filled in as an example. Replace it with YOUR content,
# then fill in documents 2–5 (or add more!).

my_documents = [
    {
        "title": "Franchise History and the Original Six Era",
        "source": "https://www.britannica.com/topic/Boston-Bruins",
        "last_updated": "2026",
        "content": (
            "The Boston Bruins, established in 1924, hold the distinction of being the first American-based team "
            "to join the National Hockey League (NHL). As one of the 'Original Six' franchises, alongside the "
            "Chicago Blackhawks, Detroit Red Wings, Montreal Canadiens, New York Rangers, and Toronto Maple Leafs—"
            "the Bruins helped define the early era of professional hockey. The team found early success, winning "
            "their first Stanley Cup in 1929 against the New York Rangers. Throughout their century-long history, "
            "the Bruins have captured a total of six Stanley Cup championships (1929, 1939, 1941, 1970, 1972, and 2011). "
            "The 1970s marked a golden age known as the 'Big Bad Bruins' era, characterized by a physical, aggressive "
            "style of play led by superstars Bobby Orr and Phil Esposito. Bobby Orr, a defenseman, revolutionized "
            "the game with his offensive capabilities, winning three consecutive Hart Trophies (MVP) from 1970 to 1972. "
            "The team also holds an impressive NHL record for 29 consecutive playoff appearances between 1968 and 1996. "
            "Iconic figures like Ray Bourque, who played 21 seasons for Boston, and Milt Schmidt, who served as a "
            "player, captain, coach, and general manager, are central to the franchise's legacy."
        )
    },
    {

        "title": "Current Roster and Leadership (2025-2026)",
        "source": "https://www.nhl.com/bruins/roster",
        "last_updated": "2026-04-21",
        "content": (
            "For the 2025-2026 season, the Boston Bruins have opted not to name a team captain, "
            "leaving the 'C' vacant for the first time since the early 2000s. Leadership is instead "
            "shared by a trio of alternate captains: David Pastrnak, Charlie McAvoy, and Hampus Lindholm. "
            "The roster has seen significant turnover, with long-time veterans Brad Marchand and "
            "Charlie Coyle no longer with the franchise. In their place, the team has leaned on "
            "new acquisitions like top-line center Elias Lindholm and defenseman Nikita Zadorov. "
            "A major mid-season trade also brought in Casey Mittelstadt from Colorado to bolster the "
            "center depth. The 2025-2026 squad is led by a new head coach, Marco Sturm, who has "
            "implemented a fast-paced system focusing on the offensive production of David Pastrnak, "
            "who remains the team's primary star after a 100-point regular season. Morgan Geekie "
            "has also emerged as a breakout scorer, leading the team in goals this year. The "
            "goaltending remains a top-tier strength, with Jeremy Swayman serving as the undisputed "
            "starter, backed up by Joonas Korpisalo. Other key roster members for this campaign "
            "include young defenseman Mason Lohrei and veteran Sean Kuraly, who returned to "
            "provide bottom-six stability and leadership in the room."
        )
    },
    {
        "title": "TD Garden Venue Operations and Guest Policies",
        "source": "https://www.tdgarden.com/plan-your-visit/a-z-guide",
        "last_updated": "2026",
        "content": (
            "TD Garden serves as the home arena for the Boston Bruins and is located in the city's North End. "
            "The venue has a seating capacity of 17,850 for NHL games. Guests visiting the arena must adhere to "
            "several strict policies designed for safety and efficiency. Most notably, TD Garden is a strictly "
            "cashless venue; all transactions for food, beverage, and merchandise must be made via credit card, "
            "debit card, or mobile payment (Apple Pay, Google Pay). For those carrying cash, 'Cash-to-Card' kiosks "
            "are located on Levels 4, 5, and 7 to convert cash into a prepaid Mastercard. The arena also enforces "
            "a rigorous bag policy: small bags, clutches, and wallets are permitted only if they do not exceed "
            "4in x 6in x 1.5in. Larger bags, such as backpacks or suitcases, are prohibited, though exceptions "
            "are made for medical or diaper bags. Entry is entirely mobile-based; fans must use the Boston Bruins "
            "or TD Garden mobile app to display their tickets, as paper tickets and screenshots are not accepted. "
            "The venue is also home to 'The ProShop,' the official team store located on the ground floor, and "
            "The Sports Museum, which spans levels 5 and 6."
        )
    },
    {
        "title": "The 2011 Stanley Cup Championship Run",
        "source": "https://records.nhl.com/playoff-summary/stanley-cup-winner?season=20102011",
        "last_updated": "2026",
        "content": (
            "The 2011 Stanley Cup championship is the most significant modern milestone for the Bruins franchise, "
            "ending a 39-year title drought. The journey to the trophy was historically difficult, as the Bruins "
            "became the first team in NHL history to win three seven-game series in a single postseason. They "
            "defeated the Montreal Canadiens in the first round, swept the Philadelphia Flyers in the second, "
            "and outlasted the Tampa Bay Lightning in the Eastern Conference Finals. In the Stanley Cup Finals, "
            "they faced the Vancouver Canucks, the league's top regular-season team. After falling behind 0-2 and "
            "later 2-3 in the series, the Bruins forced a decisive Game 7 in Vancouver. Goaltender Tim Thomas "
            "delivered one of the greatest playoff performances ever, earning the Conn Smythe Trophy as playoff MVP. "
            "In the final game, Patrice Bergeron and Brad Marchand each scored two goals to secure a 4-0 shutout victory. "
            "The championship parade in Boston drew over one million fans to celebrate the 'Crest on the Chest.' "
            "Other key players from this legendary squad included captain Zdeno Chara, Mark Recchi, and Milan Lucic."
        )
    },
    {
        "title": "Ticketing, Membership, and Fan Engagement",
        "source": "https://www.tdgarden.com/plan-your-visit/know-before-you-go",
        "last_updated": "2026",
        "content": (
            "Attending a Boston Bruins game is a high-demand experience with several layers of access. "
            "The 'Season Ticket Holder Waitlist' is the primary way for fans to secure permanent seats, "
            "reflecting the team's long-standing sellout streak at TD Garden. For individual games, tickets "
            "are available primarily through Ticketmaster and the official NHL Ticket Exchange. The Bruins "
            "offer several special programs, including the 'Student Discount' program where local college "
            "students can sign up for last-minute ticket alerts. Children under two years of age do not "
            "require a ticket but must sit on an adult’s lap. The 'Boston Bruins Foundation' is the team's "
            "charitable arm, raising millions for New England non-profits through the '50/50 Raffle' held "
            "during every home game. Fan engagement is also driven by 'Blades the Bruin,' the team's "
            "official mascot, and the '7th Settlement' fan zone. Additionally, the team honors its history through "
            "Heritage Hall, a permanent exhibit in the arena that showcases trophies, jerseys, and interactive "
            "displays about the team's century of hockey. For those traveling to games, TD Garden is located "
            "directly above North Station, providing direct access to the MBTA Orange and Green lines."
        )
    }
]

print(f"📝 Defined {len(my_documents)} documents for {MY_DOMAIN}")
for i, doc in enumerate(my_documents):
    print(f"   {i+1}. {doc['title']}")

📝 Defined 5 documents for Boston Bruins
   1. Franchise History and the Original Six Era
   2. Current Roster and Leadership (2025-2026)
   3. TD Garden Venue Operations and Guest Policies
   4. The 2011 Stanley Cup Championship Run
   5. Ticketing, Membership, and Fan Engagement


In [9]:
# ============================================
# PROCESS YOUR DOCUMENTS (just run this)
# ============================================
# Chunks your documents and prepares the search tool.

my_chunks = chunk_documents(my_documents)

print(f"📄 Loaded {len(my_documents)} documents")
print(f"✂️  Created {len(my_chunks)} chunks")
print()
for i, doc in enumerate(my_documents):
    doc_chunks = [c for c in my_chunks if c['title'] == doc['title']]
    print(f"   📝 {doc['title']} → {len(doc_chunks)} chunk(s)")

print(f"\n✅ Ready! Your agent can now search these documents.")

📄 Loaded 5 documents
✂️  Created 15 chunks

   📝 Franchise History and the Original Six Era → 3 chunk(s)
   📝 Current Roster and Leadership (2025-2026) → 3 chunk(s)
   📝 TD Garden Venue Operations and Guest Policies → 3 chunk(s)
   📝 The 2011 Stanley Cup Championship Run → 3 chunk(s)
   📝 Ticketing, Membership, and Fan Engagement → 3 chunk(s)

✅ Ready! Your agent can now search these documents.


### Step 3: Ask Your Agent Questions!

Try **3 types** of questions:

1. **Clearly answerable** — the answer is directly in one of your documents
2. **Needs multiple documents** — the agent has to search and combine info from different docs
3. **NOT in your documents** — the agent should say "I don’t know"

Replace the example questions below with questions about **your** domain.

In [10]:
# Question 1: Clearly answerable from your documents
result1 = run_rag_agent(
    "How many Stanley Cups have the Boston Bruins won, and what were the years?",
    my_tools,
    my_tool_functions
)


💬 User: How many Stanley Cups have the Boston Bruins won, and what were the years?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   🎯 ACTION: search_my_docs
      Input: {
  "query": "Stanley Cup championships wins years"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.09,
          "document": "Franchise History and the Original Six Era",
          "source": "https://www.britannica.com/topic/Boston-Bruins",
          "last_updated": "2026",
          "content": "heir first Stanley Cup in 1929 against the New York Rangers. Throughout their century-long history, the Bruins have captured a total of six Stanley Cup championships (1929, 1939, 1941, 1970, 1972...
      [...truncated for display, full data sent to agent]

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 2 | Stop reason: end_turn

🤖 FINAL ANSWER:
**OBSERVE:** I found the information needed. The first search result clearly states the 

In [11]:
# Question 2: Needs info from multiple documents
result2 = run_rag_agent(
    "Who is the current captain of the team, and what is the capacity of the arena where they play?",
    my_tools,
    my_tool_functions
)


💬 User: Who is the current captain of the team, and what is the capacity of the arena where they play?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   💭 THOUGHT:
      I'll search for information about the current captain and arena capacity for the Boston Bruins.

   🎯 ACTION: search_my_docs
      Input: {
  "query": "current captain Boston Bruins"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.05,
          "document": "The 2011 Stanley Cup Championship Run",
          "source": "https://records.nhl.com/playoff-summary/stanley-cup-winner?season=20102011",
          "last_updated": "2026",
          "content": "he final game, Patrice Bergeron and Brad Marchand each scored two goals to secure a 4-0 shutout victory. The championship parade in Boston drew over one million fans to cel...
      [...truncated for display, full data sent to agent]

   🎯 ACTION: search_my_docs
      Input: {
  "query": "arena capacity

In [12]:
# Question 3: NOT in your documents — agent should say "I don't know"
result3 = run_rag_agent(
    user_message="What is the salary of the head coach Marco Sturm for the 2025-2026 season?",
    tools=my_tools,
    tool_functions=my_tool_functions
)


💬 User: What is the salary of the head coach Marco Sturm for the 2025-2026 season?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   💭 THOUGHT:
      I'll search for information about Marco Sturm's salary for the 2025-2026 season.

   🎯 ACTION: search_my_docs
      Input: {
  "query": "Marco Sturm head coach salary 2025-2026"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.07,
          "document": "Current Roster and Leadership (2025-2026)",
          "source": "https://www.nhl.com/bruins/roster",
          "last_updated": "2026-04-21",
          "content": "place, the team has leaned on new acquisitions like top-line center Elias Lindholm and defenseman Nikita Zadorov. A major mid-season trade also brought in Casey Mittelstadt from Colorado to bolster the c...
      [...truncated for display, full data sent to agent]

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 2 | Stop reason: tool_use


In [13]:
# BONUS: Add 2 more questions of your own!

result4 = run_rag_agent(
    "Which player was acquired mid-season from Colorado to improve the team's center depth?",
    my_tools, my_tool_functions
)

result5 = run_rag_agent(
    "I only have cash; how can I buy food or a jersey at the arena during a game?",
    my_tools, my_tool_functions
)


💬 User: Which player was acquired mid-season from Colorado to improve the team's center depth?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   🎯 ACTION: search_my_docs
      Input: {
  "query": "Colorado acquired mid-season center depth"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.09,
          "document": "Current Roster and Leadership (2025-2026)",
          "source": "https://www.nhl.com/bruins/roster",
          "last_updated": "2026-04-21",
          "content": "place, the team has leaned on new acquisitions like top-line center Elias Lindholm and defenseman Nikita Zadorov. A major mid-season trade also brought in Casey Mittelstadt from Colorado to bolster the c...
      [...truncated for display, full data sent to agent]

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 2 | Stop reason: end_turn

🤖 FINAL ANSWER:
**ANSWER:**

According to the "Current Roster and Leadership (2025-2026

### ✅ Task 1 Checkpoint

Before moving on, check:

- [ ] Did your agent find the right documents for answerable questions?
- [ ] Did it cite which document the answer came from?
- [ ] Did it say "I don’t know" for the question NOT in your documents?

If yes — **Task 1 done!** 🎉

---

## 💥 Task 2: Break Your Agent (30 min)

Knowing **where** AI breaks is more valuable than knowing where it works. In this task, you’ll deliberately create situations that trip up your agent.

**Three failure modes to test:**

1. **Outdated Data** — Add an old version of a document with different facts. Which version does the agent use?
2. **Contradicting Documents** — Add a document that says the opposite of an existing one. How does the agent handle the conflict?
3. **Knowledge Gap** — Ask about something that’s plausible but NOT in any document. Does the agent admit it doesn’t know, or hallucinate?

In [14]:
outdated_doc = {
    "title": "Current Roster and Leadership (2023-2024)",  # Similar title to your current doc
    "source": "https://www.nhl.com/bruins/roster/20232024",
    "last_updated": "2023-10-11",  # <-- Older date
    "content": (
        "For the 2023-2024 season, the Boston Bruins are led by captain Brad Marchand, "
        "who was named the 27th captain in franchise history following the retirement "
        "of Patrice Bergeron. The leadership group includes alternate captains "
        "Charlie McAvoy and David Pastrnak. The roster features centers Charlie Coyle "
        "and Pavel Zacha. The head coach is Jim Montgomery, who is entering his "
        "second season with the team after a record-setting 2022-2023 campaign."
    )
}

# Rebuild with the outdated doc included
docs_with_outdated = my_documents + [outdated_doc]
# Note: Ensure chunk_documents is defined in your notebook from earlier steps
my_chunks = chunk_documents(docs_with_outdated)

print(f"⚠️ Rebuilt index with {len(docs_with_outdated)} docs ({len(my_chunks)} chunks)")
print(f"   Includes outdated document from {outdated_doc['last_updated']}")

# Ask a question that hits the conflict
print("\n🧪 Testing: Does the agent use the old captain/coach or the new ones?")
result_outdated = run_rag_agent(
    user_message="Who is the captain and the head coach of the Bruins?",
    tools=my_tools,
    tool_functions=my_tool_functions
)

⚠️ Rebuilt index with 6 docs (16 chunks)
   Includes outdated document from 2023-10-11

🧪 Testing: Does the agent use the old captain/coach or the new ones?

💬 User: Who is the captain and the head coach of the Bruins?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   🎯 ACTION: search_my_docs
      Input: {
  "query": "Bruins captain head coach"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.11,
          "document": "Current Roster and Leadership (2023-2024)",
          "source": "https://www.nhl.com/bruins/roster/20232024",
          "last_updated": "2023-10-11",
          "content": "For the 2023-2024 season, the Boston Bruins are led by captain Brad Marchand, who was named the 27th captain in franchise history following the retirement of Patrice Bergeron. The leadership gro...
      [...truncated for display, full data sent to agent]

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 2 | Sto

In [15]:
# ============================================
# FAILURE MODE 2: Contradicting Documents
# ============================================
# Add a document that directly contradicts an existing one.
# Example: two different allergy policies, or two different prices.

contradicting_doc = {
    "title": "TD Garden Security Alert",
    "source": "stadium-security-memo.pdf",
    "last_updated": "2026",  # Same year as our other 2026 docs!
    "content": (
        "SECURITY OVERRIDE: For the remainder of the 2025-2026 season, "
        "TD Garden has implemented a total bag ban. NO BAGS of any size, "
        "including small clutches or 4x6 wristlets, are permitted inside "
        "the arena. There are no lockers available on site."
    )
}

# Rebuild with the contradiction included
docs_with_contradiction = my_documents + [contradicting_doc]
my_chunks = chunk_documents(docs_with_contradiction)
print(f"⚠️ Rebuilt index with {len(docs_with_contradiction)} docs ({len(my_chunks)} chunks)")
print(f"   Includes contradicting document: {contradicting_doc['title']}")

# Ask a question that triggers the contradiction
print("\n🧪 Testing: How does the agent handle contradicting info?")
result_contradiction = run_rag_agent(
    user_message="Am I allowed to bring a small 4x6 inch clutch bag into the game?",
    tools=my_tools,
    tool_functions=my_tool_functions
)

⚠️ Rebuilt index with 6 docs (16 chunks)
   Includes contradicting document: TD Garden Security Alert

🧪 Testing: How does the agent handle contradicting info?

💬 User: Am I allowed to bring a small 4x6 inch clutch bag into the game?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   💭 THOUGHT:
      I need to search for information about bag policies and what items are allowed into the game.

   🎯 ACTION: search_my_docs
      Input: {
  "query": "bag policy clutch purse allowed into game"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.05,
          "document": "TD Garden Venue Operations and Guest Policies",
          "source": "https://www.tdgarden.com/plan-your-visit/a-z-guide",
          "last_updated": "2026",
          "content": "ard, or mobile payment (Apple Pay, Google Pay). For those carrying cash, 'Cash-to-Card' kiosks are located on Levels 4, 5, and 7 to convert cash into a prepaid Mastercard. The aren

In [16]:
# ============================================
# FAILURE MODE 3: Knowledge Gap
# ============================================
# Ask about something plausible but NOT in any document.
# A good agent says "I don't know." A bad one makes something up.

# Reset to original documents (remove outdated + contradicting)
my_chunks = chunk_documents(my_documents)
print(f"✅ Reset to original {len(my_documents)} documents\n")

# Ask questions about things NOT in your documents
print("🧪 Testing: Does the agent admit it doesn't know?")

gap_questions = [
    "What is the specific WiFi network name and password for fans at TD Garden?",
    "Which local charity did the Bruins donate to after their last home game?",
    "What is the current price of a 'Loge' level season ticket for the 2025-2026 season?",
]

for q in gap_questions:
    print(f"\nUser Question: {q}")
    result_gap = run_rag_agent(q, my_tools, my_tool_functions)
    # The output will show if the agent says "I don't know" or makes up a price/charity/password.
    print("-" * 40)

✅ Reset to original 5 documents

🧪 Testing: Does the agent admit it doesn't know?

User Question: What is the specific WiFi network name and password for fans at TD Garden?

💬 User: What is the specific WiFi network name and password for fans at TD Garden?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   🎯 ACTION: search_my_docs
      Input: {
  "query": "WiFi network password TD Garden fans"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.06,
          "document": "TD Garden Venue Operations and Guest Policies",
          "source": "https://www.tdgarden.com/plan-your-visit/a-z-guide",
          "last_updated": "2026",
          "content": "for medical or diaper bags. Entry is entirely mobile-based; fans must use the Boston Bruins or TD Garden mobile app to display their tickets, as paper tickets and screenshots are not accep...
      [...truncated for display, full data sent to agent]

- - - - - - - - - - - - - 

### 📝 Write Up Your Findings

**Double-click this cell to edit it.** Fill in your observations for each failure mode.

---

**Failure Mode 1: Outdated Data**

- What I tested: I tested the agent's ability to use the correct/current roster for the Boston Bruins. I added an outdated 2023-2024 roster and then asked the agent "Who is the captian and the head coach of the Boston Bruins?" to see if it would use the 2025-2026 roster document that I posted or not.
- What happened: In this case, the agent did not break and used the origional information that I provided (2025-2026 roster). The agent stated that there was no captian of the Boston Bruins and that the head coach is Marco Sturm, which is all correct information.
- Why it’s a problem: In a real business, this could lead to using outdated/expired information, which could result in major errors in pricing or leadership for example, which can damage a business's professional credibility and reputation.
- How I’d fix it: I would add a data aware search to fix problems like this. This would ensure that the agenent automatically prioritizes chucks of text that are the most recent.

**Failure Mode 2: Contradicting Documents**

- What I tested: I added a "Security Alert" document that claimed a total bag ban was in effect for 2026, directly contradicting the official "Venue Policy" which allows 4in x 6in bags. I asked the agent, "Am I allowed to bring a small 4x6 inch clutch bag?"
- What happened: In this case, the agent first flaged the conflict, but then picked a side. The agent came back saying that "No, you are NOT allowed to a bring a 4x6 inch clutch bag," which is incorrect according to the 2026 bag policies.
- Why it’s a problem: This is a problem because the agent is using the incorrect information. This can lead to serious misunderstandings between security and visitors, which can result in conflicts and poor customer/fan experience.
- How I’d fix it: I would fix this by using a version control. I would instruct the agent to not choose one document to use on its own. Instead the agent should inform the user of the conflicting information and suggest that they find another way to gather the correct information.

**Failure Mode 3: Knowledge Gap**

- What I tested: I asked relevant but unanswerable questions, such as the specific WiFi password for TD Garden and the current price for a Loge level season ticket, neither of which are in my provided documents.
- What happened: Because neither of the questions could be answered with the information provided in my documents, the agent responded with "I don't have information on that in our current documents."
- Why it’s a problem: If the agent were to confidently give the wrong answer, or hallucinate, this may lead to a customer making a poor financial decision in terms of ticket selection and purchase.
- How I’d fix it: I would fix this by lowering the confidence thresholds and tightening the system's guardrails to ensure that the agent stictly sticks to the information provided in the given documents.


### ✅ Task 2 Checkpoint

- [ ] Tested all 3 failure modes (outdated, contradiction, gap)
- [ ] Wrote up findings for each one
- [ ] Proposed at least one fix per failure mode

**Task 2 done!** 🎉

---

## 📊 Task 3: Score Your Agent (30 min)

Time to be a scientist. You’ll run **10 test questions** through your agent and score each answer on 4 criteria.

**Scoring Criteria:**

| Criteria | What It Means |
|----------|---------------|
| **Right Doc?** | Did the agent search and find the correct document? |
| **Correct?** | Is the answer factually accurate based on your documents? |
| **Hallucinated?** | Did the agent make up any information NOT in the docs? |
| **Cited?** | Did the agent say which document the answer came from? |

**Question mix:** Aim for ~5 answerable, ~3 partially answerable, ~2 unanswerable.

In [17]:
# ============================================
# YOUR 10 TEST QUESTIONS
# ============================================
# Make sure you're back to your original documents.
my_chunks = chunk_documents(my_documents)
print(f"✅ Using original {len(my_documents)} documents ({len(my_chunks)} chunks)\n")

test_questions = [
    # -- Answerable (5 questions) --
    "Who are the three alternate captains for the 2025-2026 season?",
    "How many Stanley Cups have the Bruins won and in what years?",
    "What is the specific bag policy size limit for TD Garden?",
    "Who is the current head coach for the 2025-2026 season?",
    "What are the names of the two Cash-to-Card kiosk locations?",

    # -- Partially Answerable (3 questions) --
    # These force the agent to combine info or infer from what's there
    "Compare the roles of Elias Lindholm and David Pastrnak on the current team.",
    "Is TD Garden a good place for a fan who only carries cash?",
    "Based on history and current leadership, who is the most important player on the team?",

    # -- Unanswerable (2 questions) --
    "What is the specific price of a beer at the Loge level concessions?",
    "What are the best public transportation options to get to the arena?",
]

# Run all 10 questions
results = []
for i, question in enumerate(test_questions, 1):
    print(f"\n{'#' * 60}")
    print(f"# QUESTION {i} of {len(test_questions)}")
    print(f"{'#' * 60}")
    result = run_rag_agent(question, my_tools, my_tool_functions)
    results.append(result)

print(f"\n\n✅ All {len(test_questions)} questions completed!")

✅ Using original 5 documents (15 chunks)


############################################################
# QUESTION 1 of 10
############################################################

💬 User: Who are the three alternate captains for the 2025-2026 season?

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
🔄 Step 1 | Stop reason: tool_use

   🎯 ACTION: search_my_docs
      Input: {
  "query": "alternate captains 2025-2026 season"
}

   👁️  OBSERVATION:
      [
        {
          "relevance_score": 0.05,
          "document": "Current Roster and Leadership (2025-2026)",
          "source": "https://www.nhl.com/bruins/roster",
          "last_updated": "2026-04-21",
          "content": "For the 2025-2026 season, the Boston Bruins have opted not to name a team captain, leaving the 'C' vacant for the first time since the early 2000s. Leadership is instead shared by a trio of alternate cap...
      [...truncated for display, full data sent to agent]

- - - - - - - - - - - - - - 

### 📝 Scorecard

**Double-click this cell to edit.** Fill in the table based on the output above.

| # | Question | Right Doc? | Correct? | Hallucinated? | Cited? |
|---|----------|:----------:|:--------:|:-------------:|:------:|
| 1 | Alternate Captions 2025-2026 | ✅ | ✅ | ❌ | ✅ |
| 2 | Stanley Cup History | ✅ | ✅ | ❌ | ✅ |
| 3 | Bag Policy | ✅ | ✅ | ❌ | ✅ |
| 4 | Current Head Coach | ✅ | ✅ | ❌ | ✅ |
| 5 | Cash-to-Card Kiosks| ✅ | ✅ | ❌ | ✅ |
| 6 | Lindholm vs Pastrnak| ✅ | ✅ | ❌ | ✅ |
| 7 | Cash-Only Advice| ✅ | ✅ | ❌ | ✅ |
| 8 | MVP | ✅ | ✅ | ❌ | ✅ |
| 9 | Price of Beer (Loge Level)| ✅ | ✅ | ❌ | ✅ |
| 10 | Public Transportation| ✅ | ✅ | ❌ | ✅ |

> **Reading the table:** ✅ = yes, ❌ = no. For "Hallucinated?" — ❌ is what you *want* (no hallucination). For the others, ✅ is what you want.

**My agent scored 10/10 on correctness.**

### Reflection

**Double-click to edit.** Answer these questions based on your scorecard:

- What patterns do you see? When does your agent do well vs. struggle?
- My agent succeeds with multi-step verification before giving me a final answer. A clear pattern of this verification emerged where my agent doesn't just give up if a primary keyword fails. for example, in Question 5 and 9, it attempted multiple variations of "kiosk locations" and "concession pricing" when the initial search returned no results. In terms of what my agent does well, it is successful with results that require looking through rules, such as bag policies or cashless conversions. Consequently, it struggles more when the documents only provide partial information, such as knowing the exact levels for the cashless conversion kiosks.

- Were there any surprising results (good or bad)?
- The most surprising "good" result was the agent’s response to Question 9 regarding beer price. Rather than giving up after two steps, my agent ran six different seach steps before ultimately deciding that it did not have enough information to provide a legitimate answer. Another surprising result was Question 10, where my agent successfully connected "North Station" to "MBTA subway access" without me explicitly defining that that connection should be made.

- If you could change one thing about your documents to improve accuracy, what would it be?
- I would add a "Price List" or "Menu" document so that questions regarding pricing could be answered by my agent. The "Venue Operations" document was great for policies, but it lacked  specific data, like food/drink costs or ticket tiers, that fans frequently ask about.

---

### ✅ Task 3 Checkpoint

- [ ] Ran all 10 test questions with visible output
- [ ] Filled in the scorecard table
- [ ] Wrote reflections on patterns and surprises

**Task 3 done!** 🎉

---

## 🏢 Task 4: Design a Real-World Knowledge Agent (20 min)

A company hires you as an AI consultant. They want a knowledge agent for their team. What would you build?

Pick a **real company or organization** (or make one up) and design a knowledge agent for them. Think about: What documents would it need? Who would use it? What happens if it’s wrong?

### 📝 Your Proposal

**Double-click this cell to edit.** Fill in each section.

---

**Company/Organization:** Delaware North & TD Garden — The hospitality and food service provider that manages operations for the arena, home to the Boston Bruins and Boston Celtics.

**Use Case:** A mobile-integrated knowledge agent for Front-of-House (FOH) staff (ushers, ticket takers, and security) to provide instant, accurate answers to guest inquiries regarding arena logistics, safety, and amenities during high-pressure game-day environments.

**Documents Needed (list 5–10):**
1. Arena A-Z Guide: Comprehensive list of fan policies (bag limits, prohibited items, smoking areas).
2. Concessions & Retail Directory: Real-time list of all food vendors, locations, menu items, and "Cash-to-Card" kiosk maps.
3. Emergency Response Protocols: Evacuation routes, medical station locations (First Aid), and "Code Blue" procedures.
4. ADA & Accessibility Manual: Procedures for assisted listening devices, sensory rooms, and wheelchair escorts.
5. Transit & Ride-Share Guide: Detailed pickup/drop-off zones for Uber/Lyft and MBTA North Station schedule integration.
6. Premium Seating & Club Rules: Specific access requirements for the NutraBolt Sport Club, Society on Staniford, and private suites.
7. Lost and Found Database: Current inventory of recovered items and the protocol for logging new lost item reports.

**Target Users:** Arena employees, including Ushers, Security Personnel, Guest Service Hub staff, and Concession Managers.

**Sample Questions the Agent Should Answer (list 3):**
1. "A fan has a heart condition and needs the closest First Aid station from Section 312—where is it and what is the fastest route?"
2. "A guest’s phone died and they have a digital-only ticket; where is the nearest 'Ticket Resolution' window?"
3. "Is there a designated 'Sensory Room' for a child who is feeling overwhelmed by the crowd noise?"

**Risk Assessment:**
- Risk level: MEDIUM
- Worst case if the agent is wrong: In an emergency (e.g., medical or fire), providing the wrong evacuation route or First Aid location could lead to physical injury or legal liability for the arena.
- Mitigation: Implement a "Life-Safety Override". Any question involving medical or security threats must trigger a prompt to use the physical radio immediately. All other data should be synced every 4 hours to ensure game-day specific info is current.

**Build vs. Buy:**
- Would you build a custom RAG system or use an existing platform (like Glean, Notion AI, etc.)? BUILD
- Why? TD Garden is a unique, singular venue with highly specific physical layouts and proprietary security protocols. A custom RAG system allows for a "Digital Twin" integration where the AI understands the exact geometry of the arena (e.g., "Level 4, behind Section 10") better than a generic corporate platform could.

**Success Metrics:**
- How would you measure if this agent is working well?
- Radio Traffic Reduction: 30% decrease in non-emergency "Where is...?" questions over the staff radio frequency.
- Guest Sentiment Score: 15% increase in "Staff Helpfulness" ratings in post-game surveys.
- Resolution Speed: FOH staff able to answer guest logistical questions in under 15 seconds without leaving their post.

### 📚 Example Proposal: Marriott Hotels Front Desk Agent

Here’s what a strong proposal looks like. Use this as a reference.

---

**Company/Organization:** Marriott International — global hotel chain with 8,000+ properties.

**Use Case:** A knowledge agent that helps front desk staff answer guest questions instantly, without flipping through binders or calling a manager.

**Documents Needed:**
1. Property-specific info (room types, amenities, floor plans, checkout times)
2. Loyalty program rules (Marriott Bonvoy tiers, points earning/redemption)
3. Local area guide (restaurants, attractions, transportation, emergency services)
4. Policies (cancellation, pet, smoking, noise complaints, late checkout)
5. Current promotions and packages (seasonal deals, corporate rates)
6. Restaurant/bar menus and hours (on-site dining)
7. Event/conference room details (capacity, AV equipment, catering options)
8. Emergency procedures (fire, medical, severe weather, security incidents)

**Target Users:** Front desk staff, concierge, phone operators (not guests directly).

**Sample Questions:**
1. "A Bonvoy Platinum member wants to know if they qualify for a suite upgrade — we’re at 80% occupancy tonight."
2. "Guest is asking about gluten-free options at our restaurant. What can we offer?"
3. "Someone wants to cancel their reservation for next week. What’s the cancellation window?"

**Risk Assessment:**
- Risk level: **MEDIUM**
- Worst case: Agent quotes an expired promotion or wrong cancellation policy → guest is overcharged or undercharged, brand damage.
- Mitigation: Daily document sync, flag answers older than 30 days, always show "verify with manager" for financial decisions.

**Build vs. Buy:**
- **Buy** — Use an enterprise platform (e.g., Glean or a hospitality-specific solution). Marriott has 8,000 properties, each with different docs. Custom RAG would be a maintenance nightmare. Need a platform that supports per-property document sets with centralized corporate policies.

**Success Metrics:**
- Answer accuracy: >90% on weekly spot-checks
- Time to answer: <30 seconds (vs. 3–5 minutes looking it up manually)
- Staff satisfaction: quarterly survey score >4/5
- Manager escalations reduced by 25% within 3 months

---

## 🎯 Submission Checklist

Before submitting, make sure:

- [ ] **Task 1:** Domain chosen, 5+ documents loaded, 3+ test questions run with visible output
- [ ] **Task 2:** All 3 failure modes tested, findings written up
- [ ] **Task 3:** 10 test questions run, scorecard filled in, reflection written
- [ ] **Task 4:** Real-world proposal completed (all sections filled)
- [ ] All code cells have been run (output is visible)
- [ ] Sharing: File → Share → "Anyone with the link can view"

**Paste your Colab link in Canvas. You’re done!** 🎉

---

## You Just Built a Knowledge Agent for YOUR Domain

Not a tutorial. Not a demo. **Your** agent, **your** documents, **your** expertise.

You loaded real information, tested whether the agent could use it accurately, found where it breaks, and designed how you'd deploy something like this in the real world. That's the full lifecycle of an AI project.

### The Stack So Far

```
Session 4: Tools          → Claude can DO things (calculator, weather)
Session 5: Real APIs      → Claude connects to the real world (Wikipedia, web)
Session 6: ReAct          → Claude can REASON through multi-step problems
Session 7: RAG            → Claude reads YOUR documents (today's homework!)
```

**Next up — Session 8: Multi-Agent Systems.** One agent is useful. A *team* of agents working together is powerful.

---

*ISOM 260: AI for Business* | [Course Website](https://isom-260.vercel.app)